In [0]:
estimated_availabilities_path = (
    "gs://market-intelligence-raw/ebay/"
    "item_details__estimated_availabilities/"
)

estimated_availabilities_df = (
    spark.read
    .option("multiLine", "false")
    .json(estimated_availabilities_path + "*.jsonl.gz")
)

print("Total rows:", estimated_availabilities_df.count())
print("Total columns:", len(estimated_availabilities_df.columns))

estimated_availabilities_df.printSchema()

In [0]:
# =====================================================
# Estimated availability cardinality per listing
# =====================================================

from pyspark.sql import functions as F

availability_counts = (
    estimated_availabilities_df
    .groupBy("_dlt_parent_id")
    .agg(
        F.count("*").alias("availability_count")
    )
)

display(
    availability_counts
    .agg(
        F.count("*").alias("parent_count"),
        F.avg("availability_count").alias("avg_records"),
        F.expr("percentile_approx(availability_count, 0.25)").alias("p25"),
        F.expr("percentile_approx(availability_count, 0.50)").alias("median"),
        F.expr("percentile_approx(availability_count, 0.75)").alias("p75"),
        F.min("availability_count").alias("min_records"),
        F.max("availability_count").alias("max_records")
    )
)

In [0]:
# =====================================================
# Validate availability → Item Details relationship
# =====================================================

item_details_path = (
    "gs://market-intelligence-raw/ebay/item_details/"
)

item_details_df = (
    spark.read
    .option("multiLine", "false")
    .json(item_details_path + "*.jsonl.gz")
)

availability_parent_ids = (
    estimated_availabilities_df
    .select(F.col("_dlt_parent_id").alias("parent_dlt_id"))
    .distinct()
)

item_parent_ids = (
    item_details_df
    .select(F.col("_dlt_id").alias("parent_dlt_id"))
    .distinct()
)

matched_parents = (
    availability_parent_ids
    .join(item_parent_ids, on="parent_dlt_id", how="inner")
    .count()
)

unmatched_parents = (
    availability_parent_ids
    .join(item_parent_ids, on="parent_dlt_id", how="left_anti")
    .count()
)

print("Availability parent IDs:", availability_parent_ids.count())
print("Matched to current Item Details:", matched_parents)
print("Not found in current Item Details:", unmatched_parents)

In [0]:
# =====================================================
# Availability status distribution
# =====================================================

display(
    estimated_availabilities_df
    .groupBy("estimated_availability_status")
    .agg(
        F.count("*").alias("record_count")
    )
    .orderBy(F.desc("record_count"))
)

In [0]:
# =====================================================
# Availability field completeness
# =====================================================

estimated_availabilities_df.agg(
    *[
        F.sum(
            F.when(F.col(c).isNull(), 1).otherwise(0)
        ).alias(f"{c}_null")
        for c in [
            "availability_threshold",
            "availability_threshold_type",
            "estimated_availability_status",
            "estimated_available_quantity",
            "estimated_remaining_quantity",
            "estimated_sold_quantity"
        ]
    ]
).show()

In [0]:
# =====================================================
# Quantity consistency
# =====================================================

quantity_check = estimated_availabilities_df.select(
    "_dlt_parent_id",
    "estimated_available_quantity",
    "estimated_remaining_quantity",
    "estimated_sold_quantity"
).withColumn(
    "invalid_quantity_relationship",
    (
        (F.col("estimated_remaining_quantity") < 0) |
        (F.col("estimated_sold_quantity") < 0) |
        (
            F.col("estimated_available_quantity").isNotNull() &
            (
                F.col("estimated_available_quantity") <
                F.col("estimated_remaining_quantity")
            )
        )
    )
)

print(
    "Invalid quantity relationships:",
    quantity_check.filter(
        F.col("invalid_quantity_relationship")
    ).count()
)

In [0]:
# =====================================================
# Blank / whitespace-only string check
# =====================================================

string_columns = [
    field.name
    for field in estimated_availabilities_df.schema.fields
    if field.dataType.simpleString() == "string"
]

blank_profile = estimated_availabilities_df.agg(*[
    F.sum(
        F.when(
            F.col(c).isNotNull() &
            (F.trim(F.col(c)) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in string_columns
])

blank_counts = blank_profile.first().asDict()

blank_data = [
    (column, int(blank_counts[column]))
    for column in string_columns
    if blank_counts[column] > 0
]

if blank_data:
    display(
        spark.createDataFrame(
            blank_data,
            ["column_name", "blank_count"]
        )
    )
else:
    print("No empty or whitespace-only string values found.")

# Estimated Availability Profile

The `estimated_availabilities` dataset contains **72,784 records** across **9 columns**:

- `_dlt_id`
- `_dlt_list_idx`
- `_dlt_parent_id`
- `availability_threshold`
- `availability_threshold_type`
- `estimated_availability_status`
- `estimated_available_quantity`
- `estimated_remaining_quantity`
- `estimated_sold_quantity`

The dataset represents estimated inventory availability information associated with Item Details listings, with `_dlt_parent_id` acting as the parent-listing relationship.

## Availability Cardinality per Listing

| Metric | Result |
|---|---:|
| Listings with availability records | 72,784 |
| Average records per listing | 1 |
| P25 | 1 |
| Median | 1 |
| P75 | 1 |
| Minimum records | 1 |
| Maximum records | 1 |

Every parent listing represented in this dataset has exactly **one estimated-availability record**.

Therefore, the observed relationship is effectively one-to-one:

```text
Item Details
    │
    │ 1
    │
    └────────── 0..1 Estimated Availability
```

The overall relationship remains optional from the Item Details perspective because listings without availability information are not represented in this child dataset.

### Estimated Availability → Item Details Relationship

| Validation | Result |
|---|---:|
| Availability parent IDs | 72,784 |
| Matched to current Item Details | 72,784 |
| Not found in current Item Details | 0 |

All 72,784 availability parent IDs successfully matched an existing Item Details record.
There are therefore 0 orphaned availability records in the current snapshot.
This confirms that `_dlt_parent_id` can be used reliably as the relationship key when integrating this dataset with Item Details.

### Availability Status Distribution

| Availability Status | Record Count | Approx. % |
|---|---:|---:|
| `IN_STOCK` | 68,699 | 94.39% |
| `OUT_OF_STOCK` | 4,085 | 5.61% |
| **Total** | **72,784** | **100%** |

The availability status field is fully populated and contains only two observed values.
The dataset is therefore strongly dominated by `IN_STOCK` listings, while approximately 5.6% of records are classified as `OUT_OF_STOCK`.

### Field Completeness

| Field | Null Count | Interpretation |
|---|---:|---|
| `availability_threshold` | 62,566 | Sparse / conditional |
| `availability_threshold_type` | 62,566 | Sparse / conditional |
| `estimated_availability_status` | 0 | 100% populated |
| `estimated_available_quantity` | 10,218 | ~86% populated |
| `estimated_remaining_quantity` | 0 | 100% populated |
| `estimated_sold_quantity` | 0 | 100% populated |

The important distinction is that the sparse threshold fields do not automatically indicate bad data.

`availability_threshold` and `availability_threshold_type` are populated for only about 14% of records. These fields are likely conditional fields that are only supplied when an availability threshold is relevant.

Therefore, we should not impute or replace these NULL values in Bronze.

The core status and quantity fields are much stronger:
- `estimated_availability_status` → 100% complete
- `estimated_remaining_quantity` → 100% complete
- `estimated_sold_quantity` → 100% complete
- `estimated_available_quantity` → approximately 86% complete

### Quantity Consistency

The quantity relationship was explicitly validated using:

- `estimated_available_quantity`
- `estimated_remaining_quantity`
- `estimated_sold_quantity`

Result:
- `Invalid quantity relationships`: 0

There are 0 records with invalid quantity relationships according to the validation rule.

This means the observed quantity fields are internally consistent in the current snapshot and do not require corrective transformations based on this check.

### Blank / Whitespace Validation

No empty or whitespace-only string values were found across the string columns.

Therefore, there is no evidence of hidden blank-string values that need to be converted to `NULL` during the initial Silver transformation.

### Key Findings
1. One availability record per listing
   All 72,784 parent IDs have exactly one availability record. The observed child cardinality is therefore 1:1, while the overall Item Details → Estimated Availability relationship remains optional.
2. Perfect parent integrity
   All 72,784 availability parent IDs matched current Item Details records, with 0 orphan records. `_dlt_parent_id` can therefore be used confidently for joining the child dataset to Item Details.
3. Availability status is highly complete
   The status field has no `NULL` values, with 94.39% `IN_STOCK` and 5.61% `OUT_OF_STOCK`. This makes it a strong analytical field for downstream availability analysis.
4. Threshold fields are conditionally populated
   `availability_threshold` and `availability_threshold_type` are sparse, with 62,566 NULL values each. This should be treated as expected conditional sparsity rather than a data-quality problem.
5. Quantity fields are reliable in the current snapshot
   Remaining and sold quantities are 100% populated, while available quantity is approximately 86% populated. The quantity validation found 0 invalid relationships.
6. No blank-string problems detected
   String profiling found no empty or whitespace-only values, so no additional blank-string cleanup is required before Silver transformation.


### Silver Layer Implications

Recommended Silver transformations:

```text
_dlt_parent_id
    → retain as parent listing relationship key

availability_threshold
    → numeric/integer, nullable

availability_threshold_type
    → categorical string, nullable

estimated_availability_status
    → categorical string

estimated_available_quantity
    → integer/long, nullable

estimated_remaining_quantity
    → integer/long

estimated_sold_quantity
    → integer/long
```

The raw values should remain available in Bronze for traceability.

In Silver, `NULL` threshold fields should remain `NULL` rather than being imputed because their sparsity appears to represent conditional source semantics, not missing data that needs correction.